# Test Generate `realisasi` & `implementation_summary` Gold Sample (RPJMN)

Lanjutan dari `test_modelhub_llm.ipynb`. Notebook ini generate prediksi boolean `realisasi (TRUE/FALSE)` dan `implementation_summary` untuk N baris pertama dari `gold_standard_sample.csv`, dicoba ke semua model kandidat di bawah, terus hasilnya dikumpulin jadi satu dataframe/CSV buat dibandingin ke ground truth.

**Model yang ditest:**
1. `watsonx-qwen3-30b-a3b-instruct-2507`
2. `Neutra-Qwen/Qwen3.6-35B-A3B`
3. `qwen3.8-27b`
4. `gemma-4-26B-A4B-it`
5. `llm_mini`
6. `telkom-ai-instruct`
7. `s0/gpt/terra`
8. `s0/gpt/luna`

In [ ]:
# %pip install -q langchain langchain-openai openai python-dotenv pandas

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("MODELHUB_LLM_API_KEY")
BASE_URL = os.getenv("MODELHUB_LLM_URL", "https://api-modelhub.airplayground.id").rstrip("/")

if not BASE_URL.endswith("/v1"):
    BASE_URL = f"{BASE_URL}/v1"

if not API_KEY:
    raise RuntimeError("MODELHUB_LLM_API_KEY belum diset di .env")

print("Base URL :", BASE_URL)
print("API key ada?", bool(API_KEY))

Base URL : https://api-modelhub.aiplayground.id/v1
API key ada? True


In [ ]:
import pandas as pd

GOLD_PATH = "gold_standard_sample.csv"
N_ROWS = 10

gold_df = pd.read_csv(GOLD_PATH)
sample_df = gold_df.head(N_ROWS).copy()

print("Total baris gold standard:", len(gold_df))
print("Baris yang dipakai buat test:", len(sample_df))
sample_df[["no", "id", "sub_indikator", "realisasi (TRUE/FALSE)"]]

Total baris gold standard: 292
Baris yang dipakai buat test: 10


,no,id,sub_indikator,realisasi (TRUE/FALSE)
0,1,12,KP 02.12.08 - Kabupaten/kota yang mendeklarasi...,TRUE
1,2,13,KP 02.09.03 - Persentase kebijakan di bidang k...,TRUE
2,3,15,"KP 02.09.04 - Persentase keberhasilan promosi,...",TRUE
3,4,19,KP 07.16.02 - Indeks Diplomasi Perlindungan WN...,TRUE
4,5,20,KP 02.01.01 - Persentase pemenuhan alutsista,TRUE
5,6,26,KP 02.10.13 - Persentase wilayah terkendali da...,TRUE
6,7,32,PP 02.12 - Efisiensi pemanfaatan air irigasi,TRUE
7,8,41,KP 07.12.01 - Indeks Kinerja Kebijakan Penerim...,TRUE
8,9,47,KP 07.14.01 - Imbal hasil (yield) SBN,TRUE
9,10,55,KP 02.10.04 - Peningkatan luas panen padi KSPP...,TRUE


In [3]:
MODELS_TO_TEST = [
    "watsonx-qwen3-30b-a3b-instruct-2507",
    "Neutra-Qwen/Qwen3.6-35B-A3B",
    "qwen3.8-27b",
    "gemma-4-26B-A4B-it",
    "llm_mini",
    "telkom-ai-instruct",
    "s0/gpt/terra",
    "s0/gpt/luna",
]

In [4]:
import json
import re

SYSTEM_PROMPT = (
    "Anda adalah asisten yang mengevaluasi status realisasi program prioritas "
    "pemerintah Indonesia (RPJMN) berdasarkan data indikator yang diberikan. "
    "Tentukan apakah program tersebut sudah terealisasi (TRUE) atau belum (FALSE), "
    "lalu tulis ringkasan implementasi singkat (2-3 kalimat) dalam Bahasa Indonesia. "
    "Jawab HANYA dengan JSON valid, tanpa teks lain, tanpa markdown code fence, dengan "
    'struktur persis: {"realisasi": "TRUE atau FALSE", "implementation_summary": "..."}'
)

def build_user_prompt(row: pd.Series) -> str:
    return (
        f"Indikator: {row['indikator']}\n"
        f"Sub-indikator: {row['sub_indikator']}\n"
        f"Satuan: {row['satuan']}\n"
        f"Baseline 2024: {row['baseline_2024']}\n"
        f"Target 2025: {row['target_2025']}\n"
        f"Target 2029: {row['target_2029']}\n"
        f"Sektor: {row['sektor']}\n"
        f"Kementerian/Lembaga: {row['kl']}\n"
        f"Keywords: {row['keywords']}\n\n"
        "Berdasarkan informasi di atas, apakah program ini sudah terealisasi? "
        "Jawab dalam format JSON yang diminta."
    )

def parse_model_output(text: str) -> dict:
    if text is None:
        return {"realisasi": None, "implementation_summary": None, "parse_error": "empty response"}
    cleaned = text.strip()
    cleaned = re.sub(r"^```(json)?", "", cleaned, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        data = json.loads(cleaned)
        return {
            "realisasi": str(data.get("realisasi", "")).strip().upper(),
            "implementation_summary": data.get("implementation_summary", ""),
            "parse_error": None,
        }
    except Exception:
        pass
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            return {
                "realisasi": str(data.get("realisasi", "")).strip().upper(),
                "implementation_summary": data.get("implementation_summary", ""),
                "parse_error": None,
            }
        except Exception as e:
            return {"realisasi": None, "implementation_summary": cleaned, "parse_error": f"json decode failed: {e}"}
    return {"realisasi": None, "implementation_summary": cleaned, "parse_error": "no json object found"}

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import time

def call_model(model_name: str, row: pd.Series, max_retries: int = 2, retry_delay: float = 2.0) -> dict:
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            chat = ChatOpenAI(
                api_key=API_KEY,
                openai_api_base=BASE_URL,
                model=model_name,
                temperature=float(os.getenv("MODELHUB_LLM_TEMPERATURE", 0.2)),
                max_tokens=int(os.getenv("MODELHUB_LLM_MAX_TOKENS", 500)),
                default_headers={
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
                },
            )
            messages = [
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=build_user_prompt(row)),
            ]
            response = chat.invoke(messages)
            parsed = parse_model_output(response.content)
            parsed["raw_response"] = response.content
            parsed["call_error"] = None
            return parsed
        except Exception as e:
            last_error = str(e)
            if attempt < max_retries:
                time.sleep(retry_delay)
    return {
        "realisasi": None,
        "implementation_summary": None,
        "parse_error": None,
        "raw_response": None,
        "call_error": last_error,
    }

In [6]:
results = []

total_calls = len(MODELS_TO_TEST) * len(sample_df)
call_count = 0

for model_name in MODELS_TO_TEST:
    for _, row in sample_df.iterrows():
        call_count += 1
        print(f"[{call_count}/{total_calls}] {model_name} -> no {row['no']}")
        pred = call_model(model_name, row)
        results.append({
            "model": model_name,
            "no": row["no"],
            "id": row["id"],
            "sub_indikator": row["sub_indikator"],
            "gold_realisasi": str(row["realisasi (TRUE/FALSE)"]).strip().upper(),
            "gold_implementation_summary": row["implementation_summary"],
            "pred_realisasi": pred["realisasi"],
            "pred_implementation_summary": pred["implementation_summary"],
            "realisasi_match": (pred["realisasi"] == str(row["realisasi (TRUE/FALSE)"]).strip().upper()) if pred["realisasi"] else False,
            "parse_error": pred.get("parse_error"),
            "call_error": pred.get("call_error"),
            "raw_response": pred.get("raw_response"),
        })

results_df = pd.DataFrame(results)
results_df.head(20)

[1/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 1
[2/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 2
[3/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 3
[4/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 4
[5/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 5
[6/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 6
[7/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 7
[8/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 8
[9/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 9
[10/80] watsonx-qwen3-30b-a3b-instruct-2507 -> no 10
[11/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 1
[12/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 2
[13/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 3
[14/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 4
[15/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 5
[16/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 6
[17/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 7
[18/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 8
[19/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 9
[20/80] Neutra-Qwen/Qwen3.6-35B-A3B -> no 10
[21/80] qwen3.8-27b -> no 1
[22/80] qwen3.8-27b

,model,no,id,sub_indikator,gold_realisasi,gold_implementation_summary,pred_realisasi,pred_implementation_summary,realisasi_match,parse_error,call_error,raw_response
0,watsonx-qwen3-30b-a3b-instruct-2507,1,12,KP 02.12.08 - Kabupaten/kota yang mendeklarasi...,TRUE,"Program berjalan, ditandai dengan penetapan al...",FALSE,Program belum terealisasi karena target pencap...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
1,watsonx-qwen3-30b-a3b-instruct-2507,2,13,KP 02.09.03 - Persentase kebijakan di bidang k...,TRUE,"Program berjalan, ditandai dengan pelaksanaan ...",FALSE,Pencapaian indikator kebijakan kerja sama pemb...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
2,watsonx-qwen3-30b-a3b-instruct-2507,3,15,"KP 02.09.04 - Persentase keberhasilan promosi,...",TRUE,"Program berjalan, ditandai dengan penetapan ol...",FALSE,Program belum terealisasi karena target pencap...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
3,watsonx-qwen3-30b-a3b-instruct-2507,4,19,KP 07.16.02 - Indeks Diplomasi Perlindungan WN...,TRUE,"Program berjalan, ditandai dengan terlaksanany...",FALSE,Program belum terealisasi karena target capaia...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
4,watsonx-qwen3-30b-a3b-instruct-2507,5,20,KP 02.01.01 - Persentase pemenuhan alutsista,TRUE,"Program berjalan, ditandai dengan pelaksanaan ...",FALSE,Program pemenuhan alutsista belum terealisasi ...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
5,watsonx-qwen3-30b-a3b-instruct-2507,6,26,KP 02.10.13 - Persentase wilayah terkendali da...,TRUE,"Program berjalan, ditandai dengan terlaksanany...",FALSE,Program belum terealisasi karena target pencap...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
6,watsonx-qwen3-30b-a3b-instruct-2507,7,32,PP 02.12 - Efisiensi pemanfaatan air irigasi,TRUE,"Program berjalan, ditandai dengan terealisasin...",FALSE,Program efisiensi pemanfaatan air irigasi belu...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
7,watsonx-qwen3-30b-a3b-instruct-2507,8,41,KP 07.12.01 - Indeks Kinerja Kebijakan Penerim...,TRUE,"Program berjalan, ditandai dengan realisasi Ta...",TRUE,Indeks Kinerja Kebijakan Penerimaan Negara men...,True,None,None,"{""realisasi"": ""TRUE"", ""implementation_summary""..."
8,watsonx-qwen3-30b-a3b-instruct-2507,9,47,KP 07.14.01 - Imbal hasil (yield) SBN,TRUE,"Program berjalan dan targetnya terpenuhi, dita...",FALSE,Program belum terealisasi karena target imbal ...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."
9,watsonx-qwen3-30b-a3b-instruct-2507,10,55,KP 02.10.04 - Peningkatan luas panen padi KSPP...,TRUE,"Program berjalan, ditandai dengan tersalurkann...",FALSE,Program peningkatan luas panen padi di Nusa Te...,False,None,None,"{""realisasi"": ""FALSE"", ""implementation_summary..."


In [7]:
summary_df = (
    results_df.groupby("model")
    .agg(
        n_rows=("no", "count"),
        n_call_error=("call_error", lambda x: x.notna().sum()),
        n_parse_error=("parse_error", lambda x: x.notna().sum()),
        realisasi_accuracy=("realisasi_match", "mean"),
    )
    .reset_index()
    .sort_values("realisasi_accuracy", ascending=False)
)

summary_df

,model,n_rows,n_call_error,n_parse_error,realisasi_accuracy
3,qwen3.8-27b,10,0,0,0.2
0,Neutra-Qwen/Qwen3.6-35B-A3B,10,0,0,0.1
6,telkom-ai-instruct,10,0,0,0.1
7,watsonx-qwen3-30b-a3b-instruct-2507,10,0,0,0.1
1,gemma-4-26B-A4B-it,10,0,0,0.0
2,llm_mini,10,0,0,0.0
4,s0/gpt/luna,10,10,0,0.0
5,s0/gpt/terra,10,10,0,0.0


> `realisasi_accuracy` di atas itu exact-match kasar buat boolean doang, bukan ukuran kualitas `implementation_summary` — itu masih perlu direview manual, apalagi mengingat concern soal ground truth cuma satu versi bukti (evidence yang benar bisa beda-beda, jadi similarity-based matching berisiko false negative).

In [8]:
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results_path = os.path.join(OUTPUT_DIR, "model_test_results_10rows.csv")
results_df.to_csv(results_path, index=False)

summary_path = os.path.join(OUTPUT_DIR, "model_test_summary_10rows.csv")
summary_df.to_csv(summary_path, index=False)

print("Disimpan ke:", results_path)
print("Disimpan ke:", summary_path)

Disimpan ke: output\model_test_results_10rows.csv
Disimpan ke: output\model_test_summary_10rows.csv
